# 心思维｜已有 TorchCREPE 与 VocalParse 音高交叉验证

本 Notebook 只读取 Google Drive 中已有的 `vocal_pitch_raw.npz`，并使用内嵌的 25 秒 VocalParse 测试样本。**不访问 GitHub、不克隆仓库、不下载模型、不运行 CREPE/VocalParse 推理，也不需要 GPU。**

只需运行下面唯一的代码单元格。结果会保存到 `TorchCREPE_Test` 文件夹。

In [ ]:
from google.colab import drive
from pathlib import Path
import json
import math
import statistics
from bisect import bisect_left, bisect_right

import numpy as np

# Drive 已挂载时出现 already mounted 属于正常提示
drive.mount('/content/drive')

# 本单元格完全自包含：不访问 GitHub、不克隆仓库、不下载模型
drive_root = Path('/content/drive/MyDrive/XinsW_Lead_Transcription/TorchCREPE_Test')
npz_path = drive_root / 'vocal_pitch_raw.npz'
track_json = drive_root / 'reference_pitch_track.json'
comparison_json = drive_root / 'vocalparse_pitch_comparison_30_55s.json'

if not npz_path.is_file():
    raise FileNotFoundError(f'Drive 中找不到已有音高文件：{npz_path}')

# 25 秒 VocalParse 固定样本（对应歌曲 30–55 秒片段）
parsed = {
  "bpm": 100,
  "words": [
    "SP",
    "如",
    "果",
    "世",
    "·",
    "界",
    "变",
    "·",
    "换",
    "太",
    "快",
    "SP",
    "当",
    "青",
    "·",
    "春",
    "SP",
    "慢",
    "慢",
    "走",
    "·",
    "远",
    "SP",
    "当",
    "容",
    "·",
    "颜",
    "不",
    "再",
    "如",
    "初",
    "·",
    "·",
    "见",
    "SP",
    "你",
    "不",
    "必",
    "问",
    "我",
    "SP",
    "爱",
    "·",
    "不",
    "·",
    "·",
    "爱",
    "SP",
    "也",
    "不"
  ],
  "pitches": [
    0,
    55,
    55,
    58,
    59,
    60,
    61,
    58,
    60,
    58,
    54,
    0,
    55,
    58,
    60,
    60,
    0,
    58,
    58,
    54,
    56,
    56,
    0,
    55,
    55,
    54,
    53,
    54,
    52,
    49,
    53,
    55,
    54,
    54,
    0,
    47,
    47,
    49,
    50,
    52,
    0,
    52,
    54,
    56,
    58,
    56,
    56,
    0,
    56,
    55
  ],
  "notes": [
    "NOTE_2",
    "NOTE_8",
    "NOTE_DOT_16",
    "NOTE_8",
    "NOTE_DOT_16",
    "NOTE_DOT_4",
    "NOTE_DOT_16",
    "NOTE_DOT_16",
    "NOTE_4",
    "NOTE_DOT_8",
    "NOTE_DOT_4",
    "NOTE_2",
    "NOTE_8",
    "NOTE_DOT_8",
    "NOTE_4",
    "NOTE_DOT_4",
    "NOTE_DOT_4",
    "NOTE_DOT_8",
    "NOTE_4",
    "NOTE_8",
    "NOTE_16",
    "NOTE_4",
    "NOTE_DOT_8",
    "NOTE_8",
    "NOTE_DOT_16",
    "NOTE_DOT_16",
    "NOTE_8",
    "NOTE_8",
    "NOTE_8",
    "NOTE_4",
    "NOTE_DOT_8",
    "NOTE_16",
    "NOTE_DOT_16",
    "NOTE_2",
    "NOTE_4",
    "NOTE_8",
    "NOTE_8",
    "NOTE_DOT_8",
    "NOTE_DOT_16",
    "NOTE_DOT_4",
    "NOTE_16",
    "NOTE_8",
    "NOTE_DOT_8",
    "NOTE_DOT_8",
    "NOTE_DOT_16",
    "NOTE_DOT_16",
    "NOTE_DOT_4",
    "NOTE_DOT_4",
    "NOTE_8",
    "NOTE_8"
  ]
}

def convert_existing_npz(path):
    with np.load(path, allow_pickle=False) as archive:
        required = {'pitch', 'periodicity'}
        missing = required.difference(archive.files)
        if missing:
            raise ValueError(f'NPZ 缺少数组 {sorted(missing)}；实际包含 {archive.files}')
        hz = np.asarray(archive['pitch']).reshape(-1).astype(np.float64)
        confidence = np.asarray(archive['periodicity']).reshape(-1).astype(np.float64)

    if len(hz) != len(confidence):
        raise ValueError('pitch 与 periodicity 数组长度不一致')
    if not np.isfinite(hz).all() or (hz < 0).any():
        raise ValueError('pitch 必须为有限非负 Hz 数值')
    if not np.isfinite(confidence).all() or ((confidence < 0) | (confidence > 1)).any():
        raise ValueError('periodicity 必须为 0 到 1 之间的有限数值')

    frames = [
        {'time_sec': i * 0.01, 'frequency_hz': float(hz[i]), 'confidence': float(confidence[i])}
        for i in range(len(hz))
    ]
    result = {
        'source_file': path.name,
        'frame_count': len(frames),
        'timing': {'source': 'confirmed_hop_seconds', 'hop_seconds': 0.01},
        'pitch_representation': 'frequency_hz',
        'frames': frames,
    }
    track_json.write_text(json.dumps(result, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    return result


DURATION_BEATS = {
    'NOTE_1': 4.0, 'NOTE_DOT_1': 6.0,
    'NOTE_2': 2.0, 'NOTE_DOT_2': 3.0,
    'NOTE_4': 1.0, 'NOTE_DOT_4': 1.5,
    'NOTE_8': 0.5, 'NOTE_DOT_8': 0.75,
    'NOTE_16': 0.25, 'NOTE_DOT_16': 0.375,
    'NOTE_32': 0.125, 'NOTE_DOT_32': 0.1875,
}


def make_events(data):
    if not (len(data['words']) == len(data['pitches']) == len(data['notes'])):
        raise ValueError('VocalParse words/pitches/notes 数量不一致')
    bpm = float(data['bpm'])
    if not math.isfinite(bpm) or bpm <= 0:
        raise ValueError('BPM 无效')

    events = []
    beat = 0.0
    for i, (word, pitch, token) in enumerate(zip(data['words'], data['pitches'], data['notes'])):
        if token not in DURATION_BEATS:
            raise ValueError(f'不支持的时值：{token}')
        if isinstance(pitch, bool) or int(pitch) != float(pitch) or not 0 <= int(pitch) <= 127:
            raise ValueError(f'无效 MIDI 音高：{pitch}')
        duration = DURATION_BEATS[token]
        pitch = int(pitch)
        rest = word == 'SP' or pitch == 0
        events.append({
            'event_id': f'vp-{i + 1:05d}',
            'raw_word': str(word),
            'lyric': None if rest or word == '·' else str(word),
            'pitch_midi': None if rest else pitch,
            'duration_token': token,
            'duration_beats': duration,
            'start_beat': beat,
            'end_beat': beat + duration,
            'is_rest': rest,
            'is_lyric_continuation': word == '·',
        })
        beat += duration
    return events


def percentile(values, fraction):
    ordered = sorted(float(value) for value in values)
    if not ordered:
        raise ValueError('percentile requires values')
    position = (len(ordered) - 1) * fraction
    lower, upper = math.floor(position), math.ceil(position)
    if lower == upper:
        return ordered[lower]
    weight = position - lower
    return ordered[lower] * (1 - weight) + ordered[upper] * weight


def compare_events(events, bpm, frames, confidence_threshold=0.5,
                   central_fraction=0.6, offset=30.0, scale=1.0):
    parsed_frames = []
    for frame in frames:
        hz = float(frame['frequency_hz'])
        midi = None if hz <= 0 else 69.0 + 12.0 * math.log2(hz / 440.0)
        parsed_frames.append((float(frame['time_sec']), midi, float(frame['confidence'])))

    parsed_frames.sort(key=lambda item: item[0])
    frame_times = [item[0] for item in parsed_frames]
    trim = (1.0 - central_fraction) / 2.0
    seconds_per_beat = 60.0 / float(bpm)
    rows, errors, rest_ratios = [], [], []

    for i, event in enumerate(events):
        start = event['start_beat'] * seconds_per_beat
        end = (event['start_beat'] + event['duration_beats']) * seconds_per_beat
        left_time = (start + (end - start) * trim) * scale + offset
        right_time = (end - (end - start) * trim) * scale + offset
        left = bisect_left(frame_times, left_time)
        right = bisect_right(frame_times, right_time)
        selected = [(pitch, conf) for _, pitch, conf in parsed_frames[left:right]]
        voiced = [(pitch, conf) for pitch, conf in selected
                  if pitch is not None and conf >= confidence_threshold]

        row = {
            'event_index': i,
            'raw_word': event['raw_word'],
            'is_lyric_continuation': event['is_lyric_continuation'],
            'start_sec': start,
            'end_sec': end,
            'reference_start_sec': start * scale + offset,
            'reference_end_sec': end * scale + offset,
            'is_rest': event['is_rest'],
            'frame_count': len(selected),
            'voiced_frame_count': len(voiced),
        }

        if event['is_rest']:
            ratio = len(voiced) / len(selected) if selected else None
            row['reference_voiced_ratio'] = ratio
            if ratio is not None:
                rest_ratios.append(ratio)
        else:
            row['vocalparse_pitch_midi'] = event['pitch_midi']
            row['reference_voiced_frame_ratio'] = len(voiced) / len(selected) if selected else None
            if voiced:
                values = [pitch for pitch, _ in voiced]
                confidences = [conf for _, conf in voiced]
                reference = float(statistics.median(values))
                q25 = percentile(values, 0.25)
                q75 = percentile(values, 0.75)
                mad = float(statistics.median(abs(pitch - reference) for pitch in values))
                error = float(event['pitch_midi']) - reference
                row.update({
                    'reference_median_pitch_midi': reference,
                    'reference_pitch_q25_midi': q25,
                    'reference_pitch_q75_midi': q75,
                    'reference_pitch_iqr_semitones': q75 - q25,
                    'reference_pitch_mad_semitones': mad,
                    'reference_pitch_min_midi': min(values),
                    'reference_pitch_max_midi': max(values),
                    'pitch_error_semitones': error,
                    'reference_confidence_mean': statistics.mean(confidences),
                    'reference_confidence_median': float(statistics.median(confidences)),
                    'is_octave_mismatch': abs(abs(error) - 12.0) <= 1.0,
                })
                errors.append((error, row['is_octave_mismatch']))
            else:
                row.update({
                    'reference_median_pitch_midi': None,
                    'reference_pitch_q25_midi': None,
                    'reference_pitch_q75_midi': None,
                    'reference_pitch_iqr_semitones': None,
                    'reference_pitch_mad_semitones': None,
                    'reference_pitch_min_midi': None,
                    'reference_pitch_max_midi': None,
                    'pitch_error_semitones': None,
                    'reference_confidence_mean': None,
                    'reference_confidence_median': None,
                    'is_octave_mismatch': None,
                })
        rows.append(row)

    absolute_errors = [abs(error) for error, _ in errors]
    note_count = sum(not event['is_rest'] for event in events)
    summary = {
        'note_count': note_count,
        'compared_note_count': len(errors),
        'uncompared_note_count': note_count - len(errors),
        'median_absolute_error_semitones': float(statistics.median(absolute_errors)) if absolute_errors else None,
        'within_one_semitone_rate': sum(error <= 1.0 for error in absolute_errors) / len(absolute_errors) if absolute_errors else None,
        'octave_mismatch_count': sum(flag for _, flag in errors),
        'rest_count': sum(bool(event['is_rest']) for event in events),
        'compared_rest_count': len(rest_ratios),
        'mean_rest_voiced_ratio': float(statistics.mean(rest_ratios)) if rest_ratios else None,
        'confidence_threshold': confidence_threshold,
        'central_fraction': central_fraction,
        'reference_time_offset_sec': offset,
        'reference_time_scale': scale,
        'comparison_type': 'model_to_model_not_ground_truth',
    }
    return {'summary': summary, 'events': rows}


converted = convert_existing_npz(npz_path)
events = make_events(parsed)
report = compare_events(
    events, parsed['bpm'], converted['frames'],
    confidence_threshold=0.5, central_fraction=0.6, offset=30.0, scale=1.0,
)

# 网格搜索：同时校准切片起点偏移和时间尺度。只作诊断，不自动改写 BPM/音符。
note_count = sum(not event['is_rest'] for event in events)
offset_values = [round(29.5 + i * 0.1, 2) for i in range(16)]
scale_values = [round(0.96 + i * 0.01, 2) for i in range(15)]
candidates = []
required_count = math.ceil(note_count * 0.8)

for candidate_offset in offset_values:
    for candidate_scale in scale_values:
        candidate_report = compare_events(
            events, parsed['bpm'], converted['frames'],
            confidence_threshold=0.5, central_fraction=0.6,
            offset=candidate_offset, scale=candidate_scale,
        )
        summary = candidate_report['summary']
        coverage = summary['compared_note_count'] / note_count if note_count else 0.0
        candidates.append({
            'reference_time_offset_sec': candidate_offset,
            'reference_time_scale': candidate_scale,
            'compared_note_count': summary['compared_note_count'],
            'note_count': note_count,
            'note_coverage': coverage,
            'median_absolute_error_semitones': summary['median_absolute_error_semitones'],
            'within_one_semitone_rate': summary['within_one_semitone_rate'],
            'mean_rest_voiced_ratio': summary['mean_rest_voiced_ratio'],
            'eligible': summary['compared_note_count'] >= required_count,
        })

eligible = [item for item in candidates
            if item['eligible'] and item['median_absolute_error_semitones'] is not None]
eligible.sort(key=lambda item: (
    item['median_absolute_error_semitones'],
    -item['note_coverage'],
    item['reference_time_offset_sec'],
    item['reference_time_scale'],
))
report['time_mapping_calibration'] = {
    'method': 'grid_search_offset_and_time_scale',
    'minimum_coverage': 0.8,
    'required_compared_note_count': required_count,
    'candidate_count': len(candidates),
    'eligible_candidate_count': len(eligible),
    'best_candidate': eligible[0] if eligible else None,
    'best_full_coverage_candidate': next(
        (item for item in eligible if item['compared_note_count'] == note_count), None
    ),
    'top_10_candidates': eligible[:10],
    'warning': '模型间拟合不是人工真值；不要据此自动改写 BPM、音符时值或音高。',
}

comparison_json.write_text(
    json.dumps(report, ensure_ascii=False, indent=2) + '\n', encoding='utf-8'
)

print('参考音高帧数：', converted['frame_count'])
print('采样间隔：0.01 秒；比较片段：歌曲 30–55 秒')
print('\n===== 交叉验证汇总 =====')
print(json.dumps(report['summary'], ensure_ascii=False, indent=2))
print('\n===== 自动时间轴校准结果（模型间拟合，不是真值） =====')
print(json.dumps(report['time_mapping_calibration'], ensure_ascii=False, indent=2))

print('\n===== 完整覆盖候选的逐音符误差与稳定性诊断 =====')
full_candidate = report['time_mapping_calibration']['best_full_coverage_candidate']
if full_candidate:
    calibrated_report = compare_events(
        events, parsed['bpm'], converted['frames'],
        confidence_threshold=0.5, central_fraction=0.6,
        offset=full_candidate['reference_time_offset_sec'],
        scale=full_candidate['reference_time_scale'],
    )
    print('采用参数：', json.dumps(full_candidate, ensure_ascii=False))
    stability_rows = []
    for event, row in zip(events, calibrated_report['events']):
        if not event['is_rest'] and row['pitch_error_semitones'] is not None:
            stability_rows.append({
                '事件序号': row['event_index'] + 1,
                '字/标记': event['raw_word'],
                '续音标记': event['is_lyric_continuation'],
                'VocalParse MIDI': row['vocalparse_pitch_midi'],
                'CREPE 中位音高': round(row['reference_median_pitch_midi'], 2),
                '偏差(半音)': round(row['pitch_error_semitones'], 2),
                '参考音高 IQR': round(row['reference_pitch_iqr_semitones'], 2),
                '参考音高 MAD': round(row['reference_pitch_mad_semitones'], 2),
                '参考音高范围': f"{row['reference_pitch_min_midi']:.1f}–{row['reference_pitch_max_midi']:.1f}",
                '有声帧比例': round(row['reference_voiced_frame_ratio'], 2),
                '平均周期性': round(row['reference_confidence_mean'], 2),
                '有效帧数': row['voiced_frame_count'],
            })
    for item in stability_rows:
        print(item)

    stable_disagreements = [
        row for row in stability_rows
        if row['参考音高 IQR'] <= 1.5 and abs(row['偏差(半音)']) > 1.5
    ]
    unstable_reference = [
        row for row in stability_rows if row['参考音高 IQR'] > 1.5
    ]
    print('\n参考轨迹较集中但与 VocalParse 偏差较大的事件（IQR≤1.5 且偏差>1.5）：', len(stable_disagreements))
    for item in stable_disagreements:
        print(item)
    print('\nTorchCREPE 参考音高波动较大的事件（IQR>1.5）：', len(unstable_reference))
    for item in unstable_reference:
        print(item)
else:
    print('当前搜索范围内没有覆盖全部音符的候选。')

print('\n===== 原始时间轴音高偏差最大的前 15 个音符（保留作对照） =====')
original_rows = []
for event, row in zip(events, report['events']):
    if not event['is_rest'] and row['pitch_error_semitones'] is not None:
        original_rows.append({
            '字': event['raw_word'],
            'VocalParse MIDI': row['vocalparse_pitch_midi'],
            'TorchCREPE MIDI中位数': round(row['reference_median_pitch_midi'], 2),
            '误差(半音)': round(row['pitch_error_semitones'], 2),
            '疑似八度错误': row['is_octave_mismatch'],
            '有效帧数': row['voiced_frame_count'],
        })
original_rows.sort(key=lambda item: abs(item['误差(半音)']), reverse=True)
for item in original_rows[:15]:
    print(item)

print('\n完整比较报告：', comparison_json)
print('说明：periodicity 阈值 0.5 仅用于筛选，不代表人工真值；本报告比较的是两个模型输出。')
